<a href="https://colab.research.google.com/github/Gulshan733/Gulshan733/blob/main/Copy_of_scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
!pip install -q spandrel
import torch; print(torch.__version__, torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
Python 3.13.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 22.6 MB/s eta 0:00:00
2.11.0+cu128 True


In [ ]:
!ls -la /content/*.mp4
!ffprobe -v error -show_entries stream=codec_type,width,height,r_frame_rate,nb_frames -of compact /content/vid1.mp4

-rw-r--r-- 1 root root 15208764 Sep 24 19:51 /content/vid1.mp4
stream|codec_type=video|width=1280|height=720|r_frame_rate=24/1|nb_frames=240
stream|codec_type=audio|r_frame_rate=0/0|nb_frames=469


In [ ]:
import os, json, subprocess, time, torch
import numpy as np
import torch.nn.functional as F
from fractions import Fraction
from spandrel import ModelLoader

video_path = "/content/vid1.mp4"
model_name = "RealESRGAN_x4plus"
TARGETS = {"4K": 2160, "1080p": 1080}   # by shorter side; one model pass feeds both

BASE = "https://github.com/xinntao/Real-ESRGAN/releases/download/"
URLS = {"RealESRGAN_x4plus": BASE + "v0.1.0/RealESRGAN_x4plus.pth",
        "realesr-general-x4v3": BASE + "v0.2.5.0/realesr-general-x4v3.pth"}
os.makedirs("/content/models", exist_ok=True)
wpath = f"/content/models/{model_name}.pth"
if not os.path.exists(wpath):
    torch.hub.download_url_to_file(URLS[model_name], wpath)
model = ModelLoader().load_from_file(wpath).cuda().eval()
dtype = torch.float16 if model.supports_half else torch.float32
model = model.to(dtype)
S = model.scale
TILE, PAD = 640, 16

def upscale(img):
    _, _, H, W = img.shape
    if max(H, W) <= TILE:
        return model(img)
    out = torch.zeros((1, 3, H * S, W * S), dtype=img.dtype, device=img.device)
    for y in range(0, H, TILE):
        for x in range(0, W, TILE):
            y0, y1 = max(y - PAD, 0), min(y + TILE + PAD, H)
            x0, x1 = max(x - PAD, 0), min(x + TILE + PAD, W)
            t = model(img[:, :, y0:y1, x0:x1])
            h, w = min(TILE, H - y), min(TILE, W - x)
            ty, tx = y - y0, x - x0
            out[:, :, y*S:(y+h)*S, x*S:(x+w)*S] = t[:, :, ty*S:(ty+h)*S, tx*S:(tx+w)*S]
    return out

p = json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
     "stream=width,height,r_frame_rate,nb_frames:stream_side_data=rotation", "-of", "json", video_path],
    capture_output=True, text=True).stdout)["streams"][0]
w, h = p["width"], p["height"]
rot = 0
for sd in p.get("side_data_list", []):
    rot = int(sd.get("rotation", 0) or 0)
if abs(rot) % 180 == 90:
    w, h = h, w
fps_str = p["r_frame_rate"]
total = int(p.get("nb_frames") or 0)

name = os.path.splitext(os.path.basename(video_path))[0]
outs = []
for label, short in TARGETS.items():
    f = short / min(w, h)
    ow, oh = int(round(w * f / 2)) * 2, int(round(h * f / 2)) * 2
    path = f"/content/{name}_upscaled_{label}_{ow}x{oh}.mp4"
    enc = subprocess.Popen(["ffmpeg", "-loglevel", "error", "-y",
                            "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{ow}x{oh}", "-r", fps_str, "-i", "-",
                            "-i", video_path, "-map", "0:v", "-map", "1:a?",
                            "-c:v", "libx264", "-preset", "slow", "-crf", "16", "-pix_fmt", "yuv420p",
                            "-c:a", "copy", "-shortest", path], stdin=subprocess.PIPE)
    outs.append((ow, oh, path, enc))
    print(f"{w}x{h} -> {ow}x{oh} ({label})")
print(f"model {model_name} ({S}x, {dtype}) | {fps_str} fps | {total} frames")

dec = subprocess.Popen(["ffmpeg", "-loglevel", "error", "-i", video_path,
                        "-f", "rawvideo", "-pix_fmt", "rgb24", "-"], stdout=subprocess.PIPE)
frame_bytes = w * h * 3
n, t0 = 0, time.time()
with torch.inference_mode():
    while True:
        buf = dec.stdout.read(frame_bytes)
        if len(buf) < frame_bytes:
            break
        x = torch.from_numpy(np.frombuffer(buf, np.uint8).reshape(h, w, 3).copy())
        x = x.cuda().permute(2, 0, 1)[None].to(dtype) / 255
        y = upscale(x)
        for ow, oh, _, enc in outs:
            z = y if y.shape[-2:] == (oh, ow) else F.interpolate(
                y.float(), size=(oh, ow), mode="bicubic", antialias=True, align_corners=False)
            z = (z.clamp(0, 1) * 255).round().byte()[0].permute(1, 2, 0).cpu().numpy()
            enc.stdin.write(z.tobytes())
        n += 1
        if n % 24 == 0:
            el = time.time() - t0
            print(f"{n}/{total} frames | {n/el:.2f} fps | ETA {(total-n)*el/n/60:.1f} min", flush=True)
dec.wait()
for _, _, path, enc in outs:
    enc.stdin.close(); enc.wait()
    assert enc.returncode == 0, f"Encoding failed for {path}"
    print("Saved:", path, f"{os.path.getsize(path)/1e6:.1f} MB")

100%|██████████| 63.9M/63.9M [00:00<00:00, 463MB/s]


1280x720 -> 3840x2160 (4K)
1280x720 -> 1920x1080 (1080p)
model RealESRGAN_x4plus (4x, torch.float16) | 24/1 fps | 240 frames
24/240 frames | 0.19 fps | ETA 19.1 min
48/240 frames | 0.19 fps | ETA 17.3 min
72/240 frames | 0.18 fps | ETA 15.3 min
96/240 frames | 0.18 fps | ETA 13.3 min
120/240 frames | 0.18 fps | ETA 11.1 min
144/240 frames | 0.18 fps | ETA 9.0 min
168/240 frames | 0.18 fps | ETA 6.7 min
192/240 frames | 0.18 fps | ETA 4.5 min
216/240 frames | 0.18 fps | ETA 2.3 min
240/240 frames | 0.18 fps | ETA 0.0 min
Saved: /content/vid1_upscaled_4K_3840x2160.mp4 41.0 MB
Saved: /content/vid1_upscaled_1080p_1920x1080.mp4 11.4 MB
